In [27]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)
os.getenv('OPENAI_API_KEY')
default_model = os.getenv('OPENAI_DEFAULT_MODEL')
print(default_model)

gpt-4o-mini


In [28]:
from openai import OpenAI

client = OpenAI()

In [19]:
message=[]
while True:
    input_msg = input('질문을 입력하세요(종료하려면 quit 입력) : ').strip()
    if input_msg.lower() == 'quit':
        print('프로그램을 종료합니다.')
        break

    message.append({'role':'user', 'content' : input_msg})

    ret = client.responses.create(
        model=default_model,
        instructions='너는 친절한 가이드야',
        input=message
    )
    message.append({'role':'assistant', 'content' : ret.output_text})
   
    ret.output_text
    print(ret.output_text)
    print('-'*50)

봄에 피는 꽃들은 다양한 색상을 가지고 있어요. 예를 들어:

- **벚꽃**: 연분홍색 또는 흰색
- **개나리**: 노란색
- **목련**: 하얀색 또는 자주색
- **튤립**: 빨간색, 노란색, 보라색 등 다양한 색상
- **제비꽃**: 보라색, 흰색 또는 노란색

봄은 정말 다채로운 색깔로 가득 찬 계절이죠! 어떤 꽃이 가장 좋아요?
--------------------------------------------------
프로그램을 종료합니다.


In [20]:
import tiktoken

default_model = "gpt-4o-mini"

text = (
    "줄바꿈 없이 숫자를 콤마(,)로 구분하고 "
    "1부터 100까지 숫자를 세줘. 예를 들면 : 1, 2, 3, ...."
)

print("원본 문자열 : ", text)

encoding = tiktoken.encoding_for_model(default_model)
tokens = encoding.encode(text)

print(f"토큰 개수: {len(tokens)}")
print(f"토큰 목록: {tokens}")
print()

models = [
    "gpt-4o-mini",
    "gpt-4-turbo",
    "gpt-3.5-turbo"
]

for model in models:
    try:
        encoding = tiktoken.encoding_for_model(model)
        tokens = encoding.encode(text)

        print(f"[{model}]")
        print(f"토큰 수: {len(tokens)}")
        print(f"처음 10개 토큰: {tokens[:10]}")
        print(f"복원 결과: {encoding.decode(tokens[:10])}")
        print("-" * 50)

    except Exception as e:
        print(f"{model} 오류")
        print(e)

원본 문자열 :  줄바꿈 없이 숫자를 콤마(,)로 구분하고 1부터 100까지 숫자를 세줘. 예를 들면 : 1, 2, 3, ....
토큰 개수: 41
토큰 목록: [78167, 16648, 164836, 111626, 151353, 89174, 79316, 97, 11630, 7, 83132, 3710, 23019, 15567, 17231, 220, 16, 42070, 220, 1353, 37046, 151353, 89174, 28126, 153545, 13, 28065, 4831, 37386, 9018, 712, 220, 16, 11, 220, 17, 11, 220, 18, 11, 23635]

[gpt-4o-mini]
토큰 수: 41
처음 10개 토큰: [78167, 16648, 164836, 111626, 151353, 89174, 79316, 97, 11630, 7]
복원 결과: 줄바꿈 없이 숫자를 콤마(
--------------------------------------------------
[gpt-4-turbo]
토큰 수: 57
처음 10개 토큰: [59269, 226, 39277, 242, 166, 123, 230, 47782, 13094, 70292]
복원 결과: 줄바꿈 없이 �
--------------------------------------------------
[gpt-3.5-turbo]
토큰 수: 57
처음 10개 토큰: [59269, 226, 39277, 242, 166, 123, 230, 47782, 13094, 70292]
복원 결과: 줄바꿈 없이 �
--------------------------------------------------


Tools -> function calling 

In [21]:
city = "서울"

ret1 = client.responses.create(
    model=default_model,
    input=f'{city}의 날씨를 알려줘'
)
print(ret1.output_text)


현재 서울의 날씨를 실시간으로 확인할 수는 없지만, 서울은 일반적으로 사계절이 뚜렷합니다. 봄(3-5월)과 가을(9-11월)에는 기온이 온화하고 날씨가 맑은 편이며, 여름(6-8월)에는 더위와 습도가 높고 장마가 있습니다. 겨울(12-2월)에는 춥고 눈이 오는 경우가 많습니다.

정확한 날씨 정보는 기상청 웹사이트나 날씨 앱을 통해 확인해 보시는 것을 추천합니다.


In [22]:
def get_weather(city):
    '''도시 이름을 입력받아서 해당 도시의 날씨를 반환하는 함수'''
    weather_data = {'서울':'맑음 24도', 
                    '부산':'흐림 22도', 
                    '도쿄':'비 20도'}
    return weather_data.get(city, '해당 도시의 날씨 정보를 찾을 수 없습니다.')

In [23]:
# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "도시 이름을 입력받아서 해당 도시의 날씨를 반환하는 함수",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "날씨를 알고 싶은 도시 이름",
                },
            },
            "required": ["city"],
        },
    },
]

In [24]:
city = "부산"
tool_map= {"get_weather": get_weather}

input_msg = [{'role':'user', 'content' : f'{city}의 오늘 날씨를 알려줘'}]

#1차 호출
ret1 = client.responses.create(
    model=default_model,
    input=input_msg,
    tools=tools,
    tool_choice="auto"
)


In [28]:
import json

tool_output = []

for item in ret1.output:
    if item.type=='function_call':
        function_name = item.name
        arguments = json.loads(item.arguments or '{}')
        result = tool_map[function_name](**arguments)
        tool_output.append({'type' :'function_call_output',
                            'call_id': item.call_id,
                            'output': result})

ret2 = client.responses.create(
    model=default_model,
    input=input_msg + ret1.output + tool_output
)
print(ret1.output_text)

arguments='{city: "부산"}' 
name='get_weather'
type= "function_call"
call_id='call_80fPUqMUSJNyfd7H1AdF9rZJ'

In [33]:
def calculate(operation: str, a: float, b: float) -> str:
    if operation == "add":
        return f"{a} + {b} = {a + b}"
    elif operation == "subtract":
        return f"{a} - {b} = {a - b}"
    elif operation == "multiply":
        return f"{a} × {b} = {a * b}"
    elif operation == "divide":
        if b != 0:
            return f"{a} ÷ {b} = {a / b}"
        return "0으로 나눌 수 없습니다."
    return "지원하지 않는 연산입니다."



In [41]:

tool_map= {"calculate": calculate}

input_msg = [{'role':'user', 'content' : "10 더하기 25는?"}]

#1차 호출
ret1 = client.responses.create(
    model=default_model,
    input=input_msg,
    tools=tools,
    tool_choice="auto"
)

In [ ]:
def process_user_request(user_message: str):
    # input 메시지 (멀티턴이면 여기에 누적하면 됩니다)
    input_messages = [{"role": "user", "content": user_message}]

    # 1차 호출: 모델이 답을 바로 하거나(function 불필요),
    #          function_call을 만들 수도 있음
    resp1 = client.responses.create(
        model=default_model,
        input=input_messages,
        tools=tools,
        tool_choice="auto",
        temperature=0,
    )

    return

In [42]:
# 1. Define a list of callable tools for the model
tools = [
{
        "type": "function",
        "name": "calculate",
        "description": "두 숫자의 사칙연산을 수행하는 함수",
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "description": "연산 종류 (add, subtract, multiply, divide)",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "a": {"type": "number", "description": "첫 번째 숫자"},
                "b": {"type": "number", "description": "두 번째 숫자"},
            },
            "required": ["operation", "a", "b"]
        }
    },
]

In [43]:
import json

tool_output = []

for item in ret1.output:
    if item.type=='function_call':
        function_name = item.name
        arguments = json.loads(item.arguments or '{}')
        result = tool_map[function_name](**arguments)
        tool_output.append({'type' :'function_call_output',
                            'call_id': item.call_id,
                            'output': result})

ret2 = client.responses.create(
    model=default_model,
    input=input_msg + ret1.output + tool_output
)
print(ret2.output_text)



10 더하기 25는 35입니다.


In [ ]:
def a(**abc):
    pass

a(a1=1, a2=2)

In [44]:
question = "서울의 오늘 날씨를 알려줘"

res = client.responses.create(
    model=default_model,
    input= question,
    tools = [{'type':'web_search_preview'}]
)
print(res.output_text)

서울의 오늘 날씨는 맑고 기온이 높을 것으로 예상됩니다. 오늘(6월 16일) 서울의 아침 기온은 약 21도에서 시작하여, 낮 최고 기온은 33도 안팎까지 오를 것으로 보입니다. 이는 평년보다 약 5도 정도 높은 기온입니다. ([news.sbs.co.kr](https://news.sbs.co.kr/news/endPage.do?news_id=N1008611087&plink=OLDURL&utm_source=openai))

오늘의 일출은 오전 5시 11분, 일몰은 오후 7시 56분입니다. ([solarwatch.app](https://solarwatch.app/ko/sunrise-sunset/south-korea/seoul?utm_source=openai))

대기 불안정으로 인해 강원 정선 등지로는 소나기가 내리고 있으며, 저녁까지 경기 북부와 동쪽 지역을 중심으로 5~10mm의 소나기가 지나겠습니다. 내일도 하늘은 대체로 맑겠고, 제주도는 오후부터 비가 조금 내릴 것으로 예상됩니다. ([news.sbs.co.kr](https://news.sbs.co.kr/news/endPage.do?news_id=N1008611087&plink=OLDURL&utm_source=openai))

따라서 오늘 하루 맑은 날씨와 높은 기온이 지속될 것으로 보입니다. 


In [45]:
# 1. Vector Store 생성
vector_store = client.vector_stores.create(
    name="PDF 저장소"
)

# 2. a.pdf 업로드 + Vector Store에 등록
file_batch = client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id,
    files=[open("Ai_Agents_Characteristics_And_Impact_Essay.pdf", "rb")]
)
print("업로드 상태:", file_batch.status)
print("파일 개수:", file_batch.file_counts)

# 3. File Search로 PDF 검색/요약
response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }],
    input="업로드한 PDF를 5줄로 요약해줘."
)

print(response.output_text)

업로드 상태: completed
파일 개수: FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)
이 PDF 문서는 AI 에이전트의 특징과 그들이 우리 삶에 미치는 영향을 다루고 있습니다. 주요 내용은 다음과 같습니다:

1. **AI 에이전트의 특징**: 자율성, 학습 능력, 적응성 등으로, 이는 다양한 분야에서 자동화와 효율성을 높입니다.
2. **긍정적인 영향**: AI는 일상생활의 편리함을 제공하고, 교육 및 의료 분야의 혁신을 촉진하여 사용자 맞춤형 서비스를 가능하게 합니다.
3. **부정적인 영향**: AI의 발전은 일자리 감소, 개인정보 문제, 기술 의존성 등의 부작용을 동반하고 있습니다.
4. **윤리적 고려**: AI 사용에 따른 윤리적 문제와 사회적 규범 필요성이 강조됩니다.
5. **미래의 방향**: 기술 발전과 윤리적 기준을 조화롭게 맞춰 AI 에이전트가 인간 사회에 기여해야 한다고 결론짓고 있습니다.


In [66]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)
os.getenv('GEMINI_API_KEY')
default_model = os.getenv('GEMINI_DEFAULT_MODEL')

In [67]:
from google import genai

client = genai.Client()

In [59]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [55]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [ ]:
response = client.models.generate_content(
    model = default_model,
    contents='바나나는 무슨 색깔이니?'
)
print(response.text)

바나나는 보통 **노란색**이에요.

하지만 익기 전에는 **초록색**이고, 너무 익으면 껍질에 **갈색**이나 **검은색** 반점이 생기기도 해요.


In [65]:
response = client.models.generate_content(
    model = default_model,
    contents='봄에 대해 알려줘',
    config = {
        'system_instruction': '너는 자바 개발자로 스프링 프레임워크 전문가야',
        'temperature': 0.2,
    }
)
print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [69]:
# 1. 이미지 파일 읽기 (바이트 데이터)
with open("chart.webp", "rb") as f:
    image_data = f.read()

# 2. 멀티모달 콘텐츠 생성 (텍스트와 이미지를 리스트로 묶어서 전달)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        "이 차트 결과를 한국어로 자세히 설명해줘. 긍정/부정 비율, 주요 키워드, 계약서 위험도까지 분석해줘.",
        {
            "inline_data": {
                "data": image_data, 
                "mime_type": "image/png"
            }
        },
    ],
)

# 3. 응답 출력
print(response.text)

이 차트는 **"부서별 직원 수(Employee Count by Department)"** 분포를 보여주는 원형 차트입니다. 각 부서가 전체 직원 수에서 차지하는 비율을 퍼센테이지로 나타내고 있습니다.

---

### 📊 차트 결과 상세 설명

1.  **전체 직원 수 분포:**
    *   **IT 부서:** 전체 직원의 **24.2%**로 가장 많은 직원을 보유하고 있습니다.
    *   **운영(Operations) 부서:** 전체 직원의 **21.2%**로 두 번째로 큰 비중을 차지합니다.
    *   **영업(Sales) 부서:** 전체 직원의 **18.2%**입니다.
    *   **마케팅(Marketing) 부서:** 전체 직원의 **15.2%**입니다.
    *   **재무(Finance) 부서:** 전체 직원의 **12.1%**입니다.
    *   **인사(Human Resources) 부서:** 전체 직원의 **9.1%**로 가장 적은 직원을 보유하고 있습니다.

2.  **주요 관찰:**
    *   **IT 부서와 운영 부서가 회사 인력의 약 45.4% (24.2% + 21.2%)를 차지하며 가장 큰 규모를 이루고 있습니다.** 이는 해당 회사에서 IT 기술 및 일상적인 운영 관리의 중요성이 높다는 것을 시사할 수 있습니다.
    *   인사 부서는 다른 핵심 부서들에 비해 상대적으로 규모가 작으며, 이는 해당 회사의 조직 구조 및 인력 운영 방식에 따라 해석될 수 있습니다.

---

### 🔍 추가 요청 사항 분석

제공된 차트는 부서별 직원 수를 정량적으로 보여주는 차트이며, 다음과 같은 요청 사항에 대한 직접적인 정보는 포함하고 있지 않습니다.

1.  **긍정/부정 비율 분석:**
    *   **이 차트에서는 긍정/부정 비율을 분석할 수 없습니다.** 이 차트는 직원들의 감성이나 만족도, 의견 등을 나타내는 데이터가 아니라, 단순히 부서별 직원 수의 비율을 보여주는 정량적인 데이터입니다. 긍정/부정 비율 분석을

In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)
os.getenv('OPENAI_API_KEY')
default_model = os.getenv('OPENAI_DEFAULT_MODEL')
print(default_model)

from openai import OpenAI

client = OpenAI()

gpt-4o-mini


In [2]:
import yt_dlp
from openai import OpenAI
from pydub import AudioSegment

client = OpenAI()

def download_audio(youtube_url, output_path="temp_audio.%(ext)s"):
    """유튜브에서 오디오만 다운로드"""
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': output_path,
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])
        info = ydl.extract_info(youtube_url, download=False)
        return ydl.prepare_filename(info).rsplit('.', 1)[0] + '.mp3'

def audio_to_text(audio_path):
    """Whisper API로 음성 → 텍스트"""
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            language="ko"  # 한국어 우선
        )
    return transcript.text


In [3]:
url = "https://www.youtube.com/watch?v=AuYTk8tZw5I"  # 여기에 URL

print("1. 유튜브 오디오 다운로드 중...")
audio_file = download_audio(url)

print("2. Whisper로 텍스트 변환 중...")
full_text= audio_to_text(audio_file)

print("\n=== 변환된 텍스트 ===")
print(full_text)

1. 유튜브 오디오 다운로드 중...
[youtube] Extracting URL: https://www.youtube.com/watch?v=AuYTk8tZw5I
[youtube] AuYTk8tZw5I: Downloading webpage


[youtube] AuYTk8tZw5I: Downloading android vr player API JSON
[info] AuYTk8tZw5I: Downloading 1 format(s): 251
[download] temp_audio.webm has already been downloaded
[download] 100% of    7.88MiB
[ExtractAudio] Destination: temp_audio.mp3
Deleting original file temp_audio.webm (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=AuYTk8tZw5I
[youtube] AuYTk8tZw5I: Downloading webpage


[youtube] AuYTk8tZw5I: Downloading android vr player API JSON
2. Whisper로 텍스트 변환 중...

=== 변환된 텍스트 ===
사우디아라비아 마지막 두바이로 향합니다 이야 테이블인데? 두바이로 향하러 갑니다 두바이로 향하러 갑니다 두바이로 향하러 갑니다 시끄러워 오랜만에 좋은데에 묵으러 왔습니다 저 할게요 여기는 아랍에미리티의 두바이입니다 제가 그동안 항상 돈 아끼고 군산맞게 생활을 했는데 요번달에 제 생일이 있다 보니까 저도 오랜만에 조금 휴식을 취하면서 돈을 쓰려고 두바이에 왔는데 제가 사실 항상 아끼면서 살다 보니까 돈 쓸 줄 잘 모르잖아요 영상에서도 많이 나왔다시피 그래서 오늘 돈을 좀 쓸 줄 아시는 다른 유튜버 분들한테 도움을 받기로 했습니다 형님 그 여행다큐 버벅스TV 라고 하시는 이제 카테레에서 유튜브를 하고 계시는 분인데 인사 한번 해주시죠 네 반갑습니다 빠니보틀 구독자 여러분 빠니보틀 채널에 출연하게 돼가지고 굉장히 영광이고 제가 빠니보틀님 900명 때부터 구독을 했던 팬이기 때문에 엊그저께 올라온 사우디 영상도 되게 15,000원짜리 방에서 주무시는 모습을 보고 되게 마음이 찢어졌거든요 그래서 오늘 좋은 곳 그리고 맛있는 거 그리고 되게 굉장한 것들 많이 보여드릴 수 있도록 영상을 준비를 하고 있으니까 기대해 주십시오 제가 혼자 두바이에 여행했으면 그냥 똑같이 7,000원짜리 호스텔에서 묵고 밥도 그냥 빵조각에 먹고 그래서 차라리 도움 받는 게 낫겠더라고요 그래서 오늘 이렇게 같이 여행을 하게 됐습니다 그 벅스님 버벅스님 채널 가서 구독도 많이 해주시고 처음부터 이렇게 말하면 안 되나? 담배는 말씀하면 안 되나? 담배 피우는 거 말씀하셔도 되죠? 여자친구한테 몰래 하신다거나 그런 거 아니죠? 제가 요즘 인도 발음 연습하고 있어요 Dirty little bit 이거 이게 실제로 두바이 경찰 학교인 거죠? 여기서 관광을 시켜주는 거죠? 여기가 두바이 경찰 아카데미라고 저희는 지금 헬기를 타

In [4]:
summary_prompt = """당신은 언어 이해 및 요약에 훈련된 고도로 숙련된 AI입니다.
다음 텍스트를 읽고 간결한 추상적 단락으로 요약했으면 합니다.
전체 텍스트를 읽을 필요 없이 토론의 요점을 이해하는 데 도움이 될 수 있는 일관되고
읽을 수 있는 요약을 제공하여 가장 중요한 요점을 유지하는 것을 목표로 합니다.
불필요한 세부 사항이나 접선 사항은 피하십시오."""

key_points_prompt = """당신은 정보를 핵심 포인트로 전달하는 데 특화된 능숙한 AI입니다.
다음 텍스트를 기반으로 논의되거나 언급된 주요 포인트를 확인하고 나열합니다.
이는 논의의 본질에 가장 중요한 아이디어, 결과 또는 주제가 되어야 합니다.
당신의 목표는 누군가가 읽을 수 있는 목록을 제공하여 이야기된 내용을 빠르게 이해하는 것입니다."""

action_items_prompt = """당신은 대화를 분석하고 행동 항목을 추출하는 데 있어 AI 전문가입니다.
본문을 검토하고 합의되거나 수행이 필요하다고 언급된 모든 작업, 과제 또는 행동을 식별하십시오.
이것들은 특정 개인에게 할당된 작업일 수도 있고 그룹이 취하기로 결정한 일반적인 행동일 수도 있습니다.
이러한 행동 항목을 명확하고 간결하게 나열하십시오."""

sentiment_prompt = """당신은 언어와 감정 분석에 전문성을 갖춘 AI로서 당신의 과제는 다음 텍스트의 감정을 분석하는 것입니다.
토론의 전체적인 톤, 사용된 언어가 전달하는 감정, 단어와 구가 사용되는 맥락을 고려하십시오.
감정이 일반적으로 긍정적인지 부정적인지 중립적인지를 표시하고 가능한 한 당신의 분석에 대해 간략한 설명을 제공하십시오."""

In [5]:
def text_extract(text, prompt):
    response = client.responses.create(
        model = default_model,
        input=[{'role':'system','content': prompt},
               {'role':'user','content': text}
               ]
    )
    return response.output_text

In [6]:
#요약
summary = text_extract(full_text, summary_prompt)

In [7]:
print(summary)

사우디아라비아에서 두바이로 여행을 떠난 유튜버가 생일을 맞아 오랜만에 여유를 즐기기 위해 호화로운 체험을 계획하고 있다. 그는 비용을 아끼며 살아온 경험으로 인해 여행에서 돈을 쓰는 데 어려움을 느끼고, 다른 유튜버의 도움을 받아 다양한 즐길 거리를 찾고 있다. 헬기 투어를 포함해 멋진 경치와 맛있는 식사를 즐기며, 돈이 많은 여행에서의 친절과 서비스의 차이를 체감하고, 여행을 통해 새로운 경험을 쌓고 있다.


In [8]:
key_point = text_extract(full_text, key_points_prompt)
action_items = text_extract(full_text, action_items_prompt)
sentiment = text_extract(full_text, sentiment_prompt)

In [9]:
print(summary)
print(key_point)
print(action_items)
print(sentiment)

사우디아라비아에서 두바이로 여행을 떠난 유튜버가 생일을 맞아 오랜만에 여유를 즐기기 위해 호화로운 체험을 계획하고 있다. 그는 비용을 아끼며 살아온 경험으로 인해 여행에서 돈을 쓰는 데 어려움을 느끼고, 다른 유튜버의 도움을 받아 다양한 즐길 거리를 찾고 있다. 헬기 투어를 포함해 멋진 경치와 맛있는 식사를 즐기며, 돈이 많은 여행에서의 친절과 서비스의 차이를 체감하고, 여행을 통해 새로운 경험을 쌓고 있다.
1. **여행 목적:** 두바이에 생일 기념 여행, 오랜만에 휴식과 소비를 하러 감.
2. **유튜버와의 협업:** 다른 유튜버 ‘버벅스TV’와 함께 여행, 도움을 요청함.
3. **두바이 경찰 아카데미:** 헬기 투어 예약, 경찰 아카데미에서 관광.
4. **헬기 체험:** 헬기 비행의 현실적 느낌과 경험에 대한 갈증을 느낌.
5. **친절의 중요성:** 많은 돈을 지불할 때 친절함과 인종차별이 덜 함.
6. **풍경 감상:** 두바이의 빌딩 숲과 팜주메라 관람.
7. **소소한 대화:** 여행 중 커피 한 잔과 쉬는 시간에 대한 이야기.
8. **식사 시간:** 마지막 저녁 식사로 분수 보기, 두바이 음식에 대한 높은 만족도.
9. **비교적 저렴한 여행:** 두바이 이후 일반적인 저렴한 식사로 돌아갈 계획.
대화에서 추출한 행동 항목은 다음과 같습니다:

1. **여행 계획**:
   - 두바이 여행 준비 및 실행.
  
2. **영적 지원**:
   - 다른 유튜버(버벅스TV)에게 도움 요청 및 함께 여행하기.

3. **헬기 투어 예약**:
   - 헬기 투어 예약 및 참가.

4. **경험 공유**:
   - 두바이에서의 경험을 영상으로 기록하고 공유하기.

5. **지식 습득**:
   - 헬기 비행 교육 이수.

6. **식사 계획**:
   - 저녁 식사 장소 예약 및 먹기.

7. **여행 소감**:
   - 다양한 음식의 맛 평가 및 비교.

8. **관광 활동**:
   - 팜주메라 및 고층 빌딩 탐방.

각 항목은 향후 행동에 필요

In [10]:
news_data = {
    'summary':summary,
    'key_point':key_point,
    'action_items':action_items,
    'sentiment':sentiment
}

In [11]:
from docx import Document
doc = Document()

for key, value in news_data.items():
  heading = ' '.join(word.capitalize() for word in key.split('_'))
  doc.add_heading(heading, level=1)
  doc.add_paragraph(value)
  doc.add_paragraph() #빈 줄 추가

doc.save('test.docx')

In [ ]:
response = client.responses.create(
    model = default_model,
    input=[{'role':'system','content': '주어진 텍스트를 이모지로 변환하세요, 일반 텍스트를 사용하면 안돼'},
            {'role':'user','content': '졸립고, 덥고, 어렵고, 집 가서 자고 싶어, 맛있는 물회 먹고 싶어'}
            ]
)
print(response.output_text)

😴🔥😩🏠💤🥵🍲🌊🏖️🍽️✨


Gradio

In [17]:
import gradio as gr

def func(name):
    return f'Hello, {name}!'

demo= gr.Interface(fn = func,
             inputs='text',
             outputs='text')
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [18]:
def greet(name, is_morning, price):
    greeting = f'좋은 아침입니다' if is_morning else '좋은 저녁입니다'
    greeting += f'{name}님!'
    return greeting, price * 3

demo = gr.Interface(fn=greet,
             inputs=['text', 'checkbox', gr.Slider(0,100)],
             outputs=['text', 'number'])
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\myWork\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [ ]:
tab1 = gr.Interface(lambda x : x[::-1], inputs='text', outputs='text', title ='뒤집기')
tab2 = gr.Interface(lambda x : x.upper(), inputs='text', outputs='text', title = '대문자' )

demo = gr.TabbedInterface([tab1, tab2],['뒤집기','대문자'])
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\myWork\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\myWork\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\myWork\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\myWork\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [22]:
demo.close()

Closing server running on port: 7861


In [29]:
import gradio as gr
import random
import time

# 응답 생성 함수
def respond(msg, chat_history):
    # 미리 정해진 문장 중 하나를 랜덤 선택
    response = random.choice(['안녕하세요', '반갑습니다', '무엇을 도와드릴까요?'])

    # 유저 메시지 먼저 히스토리에 추가
    chat_history = chat_history + [{"role": "user", "content": msg}]

    # # 응답(assistant)를 한 글자씩 조금씩 출력 (스트리밍 효과)
    stream = ""
    for ch in response:
        stream += ch 
        time.sleep(0.05)    # 출력 속도 조절을 위한 지연
        # 현재까지의 대화 내역과 사용자의 입력을 반환
        # yield를 사용해 부분 출력으로 스트리밍 구현
        yield chat_history + [{"role": "assistant", "content": stream}], ""

# Gradio Blocks UI 생성
with gr.Blocks() as bl:
    chatbot = gr.Chatbot()  # 대화 내용을 보여주는 컴포넌트 생성
    msg = gr.Textbox(placeholder="메세지를 입력하세요...", label="입력") 
    clear_btn = gr.Button("초기화")# 대화창 초기화 버튼

    # 사용자가 메시지 입력 후 엔터를 누르면 respond 함수 실행
    # 입력: msg(Textbox), chatbot(Chatbot)
    # 출력: chatbot, msg (출력은 다시 텍스트박스로 전달해 입력란 비움)
    msg.submit(respond, inputs=[msg, chatbot], outputs=[chatbot, msg])

    # 초기화 버튼 클릭 시 chatbot의 내용을 빈 리스트로 초기화
    clear_btn.click(lambda: ([], ""), inputs=None, outputs=[chatbot, msg])

# 비동기 요청 처리 대기열 활성화 yield를 사용
bl.queue().launch(share=True)

# 앱 실행 및 외부 접속 가능하도록 공유 URL 생성


* Running on local URL:  http://127.0.0.1:7863

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
